# EDA: calidad del aire en Beijing (UCI 501)

Análisis exploratorio para decidir **cómo definir el problema y qué cuidar en el modelado**.
Los datos se descargan con `uv run python -m proyecto_1_esp_data.data.download`.
Este notebook se guarda **sin salidas**; para verlas, ejecútalo (`Run All`).

Preguntas que responde:
1. ¿Qué tan completos y confiables son los datos?
2. ¿Cómo se comporta PM2.5 en el tiempo (año, mes, hora)?
3. ¿Cuánto cambian las estaciones entre sí?
4. ¿Qué variables se relacionan con el PM2.5 de mañana?
5. ¿Qué tan fuerte es la línea base "mañana = hoy"?

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Subir hasta la raíz del repo (donde está pyproject.toml): evita rutas absolutas
raiz = Path.cwd()
while not (raiz / "pyproject.toml").exists():
    raiz = raiz.parent
os.chdir(raiz)

from proyecto_1_esp_data import config
from proyecto_1_esp_data.data import prepare
from proyecto_1_esp_data.data.contract import validar

plt.rcParams.update({"figure.figsize": (9, 3.8), "axes.grid": True, "grid.alpha": 0.3})
df = prepare.cargar_crudo()
df["fecha"] = pd.to_datetime(df[["year", "month", "day"]])
print(df.shape)
df.head()

## 1. Calidad y completitud

Primero se verifica el contrato de datos y se miran los nulos.

In [ ]:
print("Problemas del contrato:", validar(df))
nulos = (df.drop(columns="fecha").isna().mean() * 100).sort_values(ascending=False)
nulos[nulos > 0].plot.barh(title="% de nulos por columna").invert_yaxis()
plt.xlabel("% de filas")
plt.show()

In [ ]:
por_estacion = df.groupby("station")[["PM2.5", "CO"]].agg(
    lambda s: s.isna().mean() * 100
)
por_estacion.round(1).sort_values("CO", ascending=False)

**Lectura:** el contrato se cumple y el máximo de nulos por columna es 4,9 % (CO). Los huecos
no se reparten parejo: CO llega a 9,1 % en Dongsi. Por eso la agregación diaria exige al menos
18 horas válidas y el relleno se limita a 3 horas.

## 2. Distribución de PM2.5

In [ ]:
pm = df["PM2.5"].dropna()
print(pm.describe().round(1))
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
pm.plot.hist(bins=60, ax=ax[0], title="PM2.5 horario (µg/m³)")
ax[0].axvline(config.UMBRAL_PM25, color="red", ls="--", label="umbral")
ax[0].legend()
pm.clip(lower=1).plot.hist(
    bins=60, ax=ax[1], logx=True, title="Misma variable, eje logarítmico"
)
plt.tight_layout()
plt.show()

**Lectura:** la distribución es muy sesgada a la derecha: mediana 55 µg/m³, pero el percentil 95 es 242 y
el máximo 999. Los extremos son episodios reales de contaminación, no errores, así que no se eliminan.
Por eso conviene un target binario por umbral en vez de predecir el valor exacto.

## 3. Estacionalidad: año, mes y hora

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
df.groupby("month")["PM2.5"].mean().plot(
    marker="o", ax=ax[0], title="PM2.5 medio por mes"
)
ax[0].set_xticks(range(1, 13))
df.groupby("hour")["PM2.5"].mean().plot(
    marker="o", ax=ax[1], title="PM2.5 medio por hora del día"
)
plt.tight_layout()
plt.show()

**Lectura:** hay estacionalidad fuerte. Diciembre es el peor mes (104,6 µg/m³ de media) y agosto el mejor
(53,5). Invierno y otoño son peores que verano. También hay ciclo diario: la hora más limpia es la 7 (73,3)
y la peor la 22 (88,9).
**Consecuencia:** al medir drift hay que comparar ventanas de 12 meses completos; si no, el cambio de estación
parecería un cambio de datos.

## 4. Tabla diaria, target y balance de clases

In [ ]:
tabla = prepare.construir_tabla_modelo(df)
partes = prepare.dividir(tabla)
resumen = pd.DataFrame(
    {
        nombre: {"filas": len(p), "% días malos": round(p["target"].mean() * 100, 1)}
        for nombre, p in partes.items()
    }
).T
resumen

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
(tabla.groupby("station")["target"].mean() * 100).sort_values().plot.barh(
    ax=ax[0], title="% de días sobre el umbral, por estación"
)
(tabla.groupby("estacion_ano")["target"].mean() * 100).reindex(
    ["invierno", "primavera", "verano", "otono"]
).plot.bar(ax=ax[1], title="% de días sobre el umbral, por estación del año", rot=0)
plt.tight_layout()
plt.show()

**Lectura:**
- Las clases están razonablemente balanceadas (alrededor de 40 % de días malos), así que no hace falta
  remuestrear, pero la proporción **baja con el tiempo**: 43,6 % en train, 39,2 % en validación y 36,6 % en
  producción. Eso es un cambio real de la distribución del target.
- Las estaciones difieren (de 31,7 % en Dingling a 43,5 % en Dongsi), por lo que las métricas se reportan por estación.
- Invierno tiene 45,6 % de días malos y verano 34,4 %.

## 5. ¿Qué se relaciona con el PM2.5 de mañana?

In [ ]:
dia = prepare.tabla_diaria(df).sort_values(["station", "fecha"])
dia["pm25_manana"] = dia.groupby("station")["pm25"].shift(-1)
cols = [
    "pm25",
    "PM10",
    "SO2",
    "NO2",
    "CO",
    "O3",
    "TEMP",
    "PRES",
    "DEWP",
    "RAIN",
    "WSPM",
]
corr = (
    dia[["pm25_manana", *cols]].corr()["pm25_manana"].drop("pm25_manana").sort_values()
)
corr.plot.barh(title="Correlación con el PM2.5 de mañana")
plt.show()
corr.round(2)

In [ ]:
viento = dia.assign(rango=pd.cut(dia["WSPM"], [0, 1, 2, 3, 20]))
viento.groupby("rango", observed=True)["pm25"].mean().plot.bar(
    title="PM2.5 medio según velocidad del viento (m/s)", rot=0
)
plt.show()
(tabla.groupby("wd")["target"].mean() * 100).sort_values().round(0)

**Lectura:**
- El mejor predictor es el propio PM2.5 de hoy (correlación 0,56), seguido de NO2, PM10 y CO (0,5 a 0,53).
  Por eso los rezagos de PM2.5 irán como features.
- El viento fuerte limpia el aire: con menos de 1 m/s el PM2.5 medio es 127 µg/m³ y con más de 3 m/s es 31.
  La velocidad del viento correlaciona -0,37 con el PM2.5 de mañana.
- La dirección importa: con viento del NW solo 23 % de los días son malos; con viento del SSW, 59 %.
- Temperatura y ozono se relacionan de forma negativa y débil (-0,18 y -0,16).
- Las estaciones están muy correlacionadas entre sí (correlación diaria mínima 0,88): no son 12 muestras
  independientes, y por eso el split corta por fecha.

## 6. Línea base: "mañana = hoy"

In [ ]:
for nombre, p in partes.items():
    print(nombre, prepare.metricas(p["target"], prepare.baseline_persistencia(p)))

In [ ]:
prepare.metricas_por_estacion(
    partes["val"], prepare.baseline_persistencia(partes["val"])
).round(3)

**Lectura:** la línea base detecta cerca de 64 de cada 100 días malos en validación y 59 en producción,
con una precisión parecida. Cualquier modelo tiene que superar esos números, sobre todo en sensibilidad, que es
la métrica principal.

## Conclusiones para el modelado
1. Target binario "PM2.5 de mañana > 75 µg/m³" sobre una tabla diaria por estación.
2. Split temporal con fechas fijas; comparar ventanas de 12 meses completos.
3. Features: PM2.5 de hoy y rezagos, otros contaminantes, viento (velocidad y dirección), temperatura, estación del año.
4. Reportar métricas por estación y vigilar el cambio de proporción de días malos (43,6 % a 36,6 %).
5. Superar la línea base de persistencia (sensibilidad 0,64 en validación).